In [0]:
create schema data_modelling_demo.gold_star

In [0]:
%python
spark.sql("""
CREATE OR REPLACE TABLE data_modelling_demo.gold_star.fact_sales AS
SELECT 
  o.order_id,
  o.customer_id,
  o.order_date,
  o.order_status,
  o.payment_id,
  o.order_total,
  oi.order_item_id,
  oi.product_id,
  oi.quantity,
  oi.unit_price,
  oi.discount_amount,
  oi.line_total
FROM data_modelling_demo.silver.orders o
JOIN data_modelling_demo.silver.order_items oi
  ON o.order_id = oi.order_id
""")

DataFrame[num_affected_rows: bigint, num_inserted_rows: bigint]

In [0]:
table data_modelling_demo.gold_star.fact_sales

order_id,customer_id,order_date,order_status,payment_id,order_total,order_item_id,product_id,quantity,unit_price,discount_amount,line_total
501,C001,2024-04-01,Completed,401,1079.98,602,303,1,79.99,0.0,79.99
502,C002,2024-04-02,Completed,402,899.99,603,302,1,899.99,0.0,899.99
501,C001,2024-04-01,Completed,401,1079.98,601,301,1,999.99,20.0,979.99


In [0]:
CREATE OR REPLACE TABLE data_modelling_demo.gold_star.dim_customers AS
SELECT
  cu.customer_id,
  cu.customer_name,
  cu.email,
  cu.phone,
  c.city_name,
  s.state_name,
  co.country_name
FROM data_modelling_demo.silver.customers cu
JOIN data_modelling_demo.silver.cities c
  ON cu.city_id = c.city_id
JOIN data_modelling_demo.silver.states s
  ON c.state_id = s.state_id
JOIN data_modelling_demo.silver.countries co
  ON s.country_id = co.country_id

num_affected_rows,num_inserted_rows


In [0]:
table data_modelling_demo.gold_star.dim_customers

customer_id,customer_name,email,phone,city_name,state_name,country_name
C001,John Smith,john@gmail.com,9876543210,New York City,New York,USA
C002,Alice Johnson,alice@gmail.com,9123456780,San Francisco,California,USA
C003,David Miller,david@gmail.com,9988776655,Dallas,Texas,USA


In [0]:
CREATE OR REPLACE VIEW data_modelling_demo.gold_star.dim_payment AS
SELECT * FROM data_modelling_demo.silver.payments

In [0]:
CREATE OR REPLACE TABLE data_modelling_demo.gold_star.dim_date AS
WITH date_range AS (
  SELECT EXPLODE(SEQUENCE(TO_DATE('2024-01-01'), TO_DATE('2026-12-31'), INTERVAL 1 DAY)) AS date
)
SELECT 
  date AS date_id,
  date,
  YEAR(date) AS year,
  MONTH(date) AS month,
  DAYOFMONTH(date) AS day,
  DAYOFWEEK(date) AS day_of_week,
  DAYOFYEAR(date) AS day_of_year,
  WEEKOFYEAR(date) AS week_of_year,
  QUARTER(date) AS quarter,
  DATE_FORMAT(date, 'EEEE') AS weekday_name,
  DATE_FORMAT(date, 'MMMM') AS month_name,
  CASE WHEN DAYOFWEEK(date) IN (1, 7) THEN TRUE ELSE FALSE END AS is_weekend,
  -- Fiscal year assuming fiscal year starts in July (month 7)
  CASE 
    WHEN MONTH(date) >= 7 THEN YEAR(date) + 1
    ELSE YEAR(date)
  END AS fiscal_year,
  CASE 
    WHEN MONTH(date) >= 7 THEN MONTH(date) - 6
    ELSE MONTH(date) + 6
  END AS fiscal_month,
  CASE 
    WHEN MONTH(date) IN (1, 2, 3) THEN 'Q1'
    WHEN MONTH(date) IN (4, 5, 6) THEN 'Q2'
    WHEN MONTH(date) IN (7, 8, 9) THEN 'Q3'
    ELSE 'Q4'
  END AS quarter_name,
  LAST_DAY(date) AS last_day_of_month,
  CASE WHEN date = LAST_DAY(date) THEN TRUE ELSE FALSE END AS is_last_day_of_month
FROM date_range

num_affected_rows,num_inserted_rows


In [0]:
CREATE OR REPLACE TABLE data_modelling_demo.gold_star.dim_products AS
SELECT 
  p.product_id,
  p.product_name,
  p.unit_price,
  p.is_active,
  b.brand_name,
  b.brand_description,
  s.subcategory_name,
  c.category_name,
  c.category_description
FROM data_modelling_demo.silver.products p
JOIN data_modelling_demo.silver.brands b
  ON p.brand_id = b.brand_id
JOIN data_modelling_demo.silver.subcategories s
  ON p.subcategory_id = s.subcategory_id
JOIN data_modelling_demo.silver.categories c
  ON s.category_id = c.category_id

num_affected_rows,num_inserted_rows


In [0]:
SELECT * FROM data_modelling_demo.gold_star.dim_products LIMIT 10

product_id,product_name,unit_price,is_active,brand_name,brand_description,subcategory_name,category_name,category_description
301,iPhone 15,999.99,true,Apple,Premium Electronics,Mobiles,Electronics,Electronic Devices
302,Galaxy S24,899.99,true,Samsung,Consumer Electronics,Mobiles,Electronics,Electronic Devices
303,Nike Hoodie,79.99,true,Nike,Sportswear,Men Clothing,Fashion,Clothing and Apparel


In [0]:
SELECT 
  d.year,
  p.category_name,
  COUNT(DISTINCT f.order_id) AS total_orders,
  COUNT(f.order_item_id) AS total_items_sold,
  SUM(f.quantity) AS total_quantity,
  ROUND(SUM(f.line_total), 2) AS total_sales,
  ROUND(SUM(f.discount_amount), 2) AS total_discounts,
  ROUND(AVG(f.unit_price), 2) AS avg_unit_price,
  ROUND(AVG(f.line_total), 2) AS avg_line_total
FROM data_modelling_demo.gold_star.fact_sales f
JOIN data_modelling_demo.gold_star.dim_products p
  ON f.product_id = p.product_id
JOIN data_modelling_demo.gold_star.dim_date d
  ON f.order_date = d.date
GROUP BY d.year, p.category_name
ORDER BY d.year, total_sales DESC

year,category_name,total_orders,total_items_sold,total_quantity,total_sales,total_discounts,avg_unit_price,avg_line_total
2024,Electronics,2,2,2,1879.98,20.0,949.99,939.99
2024,Fashion,1,1,1,79.99,0.0,79.99,79.99
